# Lab 3: Decision Tree Classification  Telco Customer Churn

- Name:Teesha kumari
- Student ID:023-23-0195

- Dataset: Telco Customer Churn (cleaned in Lab 2)




In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)


plt.rcParams['figure.figsize'] = (6, 4)


## 1. Problem Definition & Dataset Verification




In [ ]:

df = pd.read_csv("clean_churn.csv")


print("Shape (rows, columns):", df.shape)
print()
print("Column dtypes:")
print(df.dtypes)
print()
print("Missing values per column:")
print(df.isnull().sum())


In [ ]:

df['Churn'].value_counts()


**Task 2  Restate the problem (1–2 sentences):**




## 2. Feature & Target Preparation



In [ ]:

if df['Churn'].dtype == object:
    y = df['Churn'].map({'Yes': 1, 'No': 0})
else:
    y = df['Churn']

drop_cols = ['Churn']
if 'customerID' in df.columns:
    drop_cols.append('customerID')

X = df.drop(columns=drop_cols)
X.head()


In [ ]:

categorical_cols = X.select_dtypes(include='object').columns.tolist()
print("Categorical columns to encode:", categorical_cols)


X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

X.head()


**Task 4 justification:**
We used one-hot encoding (`pd.get_dummies`) instead
of manual integer mapping because most categorical columns (like
`InternetService` or `PaymentMethod`) have no inherent order assigning them
numbers like 0, 1, 2 would incorrectly imply a ranking, and a Decision Tree
could split on that false ordering. One-hot encoding avoids this by giving
each category its own independent binary column.


In [ ]:

print("Any non-numeric columns left?", X.select_dtypes(include='object').columns.tolist())
print("Any missing values left?\n", X.isnull().sum().sum())
print("Final X shape:", X.shape)


## 3. Train/Test Split


In [ ]:
##Task 6
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


**Task 6  Why `stratify=y`?**

 In Lab 2 we found the `Churn` classes are
imbalanced (far more "No" than "Yes"). A plain random split could, by chance,
put too few churners in the test set, making evaluation unreliable.
`stratify=y` forces both the train and test sets to keep the same
Yes/No churn ratio as the full dataset.


In [ ]:
# Task 7: report shapes
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)


## 4. Baseline Decision Tree

 Train a completely unrestricted Decision Tree (no
max_depth limit) as a starting point / baseline to compare everything else
against.


In [ ]:
# Task 8: baseline model
baseline_model = DecisionTreeClassifier(random_state=42)
baseline_model.fit(X_train, y_train)

y_train_pred_baseline = baseline_model.predict(X_train)
y_test_pred_baseline  = baseline_model.predict(X_test)

print("Baseline tree actual depth:", baseline_model.get_depth())


**Task 9  Does the depth surprise you?**
 Write down the depth number your run printed above,
and comment on whether it feels large. An unrestricted tree will keep
splitting until each leaf is (almost) pure, which usually produces a
surprisingly deep tree — this is a strong early hint that it may be
overfitting, which we investigate properly in Part 6.


## 5. Model Evaluation




In [ ]:
# Task 10: metrics on the TEST set
acc  = accuracy_score(y_test, y_test_pred_baseline)
prec = precision_score(y_test, y_test_pred_baseline)
rec  = recall_score(y_test, y_test_pred_baseline)
f1   = f1_score(y_test, y_test_pred_baseline)

metrics_table = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-score'],
    'Value':  [acc, prec, rec, f1]
})
metrics_table


In [ ]:
# Task 11: confusion matrix
cm = confusion_matrix(y_test, y_test_pred_baseline)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Churn', 'Churn'])
disp.plot(cmap='Blues')
plt.title("Baseline Decision Tree — Confusion Matrix (Test Set)")
plt.show()

print(classification_report(y_test, y_test_pred_baseline, target_names=['No Churn', 'Churn']))


**Task 11  Read the confusion matrix:**

 Look at the bottom row of your confusion matrix
(actual churners). The bottom-right cell = churners correctly caught
(True Positives); the bottom-left cell = churners the model MISSED
(False Negatives). Write the actual numbers from your run and comment on
how many the model missed.

**Task 12  Which metric do you trust most, and why?**

 Because `Churn` is imbalanced, **recall** on the
churn class matters most here from a business standpoint, missing an
actual churner (false negative) is usually more costly than a false alarm,
since a missed churner is lost revenue nobody tried to retain. Accuracy
alone would look fine even if the model never caught a single churner.


## 6. Overfitting Investigation




In [ ]:
# Task 13: baseline train vs test accuracy
train_acc_baseline = accuracy_score(y_train, y_train_pred_baseline)
test_acc_baseline  = accuracy_score(y_test, y_test_pred_baseline)

print(f"Baseline TRAIN accuracy: {train_acc_baseline:.4f}")
print(f"Baseline TEST  accuracy: {test_acc_baseline:.4f}")
print(f"Gap: {train_acc_baseline - test_acc_baseline:.4f}")


**Task 13  Is there a large gap?**

An unrestricted tree very often scores close to
100% on the training set but noticeably lower on the test set. A large gap
like this suggests the model has memorized quirks/noise specific to the
training rows rather than learning patterns that generalize — the classic
sign of overfitting.


In [ ]:
# Task 14: train/test accuracy at multiple max_depth values
depths = [2, 3, 4, 5, 6, 8, 10, None]
results = []

for d in depths:
    model_d = DecisionTreeClassifier(max_depth=d, random_state=42)
    model_d.fit(X_train, y_train)

    train_acc = accuracy_score(y_train, model_d.predict(X_train))
    test_acc  = accuracy_score(y_test, model_d.predict(X_test))

    results.append({
        'max_depth': 'None (unrestricted)' if d is None else d,
        'train_accuracy': train_acc,
        'test_accuracy': test_acc
    })

depth_results = pd.DataFrame(results)
depth_results


In [ ]:
# Task 15: plot train vs test accuracy against max_depth

plot_depths = [d if d is not None else 12 for d in depths]

plt.plot(plot_depths, depth_results['train_accuracy'], marker='o', label='Train Accuracy')
plt.plot(plot_depths, depth_results['test_accuracy'], marker='o', label='Test Accuracy')
plt.xlabel('max_depth (12 = unrestricted)')
plt.ylabel('Accuracy')
plt.title('Train vs Test Accuracy by Tree Depth')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


**Task 15  At roughly what depth does overfitting start?**

 Look at your table/plot: train accuracy keeps
climbing (or stays high) as depth increases, but test accuracy typically
rises, peaks, then flattens or drops. Name the approximate depth (from YOUR
numbers) where the two lines start clearly separating — that's roughly
where the model begins fitting noise in the training data instead of
general patterns.


## 7. Model Selection, Feature Importance & Interpretation



In [ ]:
# Task 16: choose your best depth based on the Part 6 table.


print("Chosen max_depth:", best_depth)


**Task 16 justification:**
We chose `max_depth = <your number>` because it
gives close to the best test accuracy in the Part 6 table while keeping the
train test accuracy gap small  deeper trees kept improving training
accuracy but test accuracy stopped improving (or got worse), which is a sign
of overfitting rather than genuine improvement.


In [ ]:
# Task 17: retrain at chosen depth (default gini criterion) and evaluate fully
final_model_gini = DecisionTreeClassifier(max_depth=best_depth, random_state=42)
final_model_gini.fit(X_train, y_train)
y_pred_gini = final_model_gini.predict(X_test)

final_metrics_gini = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-score'],
    'Baseline':      [acc, prec, rec, f1],
    'Chosen depth (gini)': [
        accuracy_score(y_test, y_pred_gini),
        precision_score(y_test, y_pred_gini),
        recall_score(y_test, y_pred_gini),
        f1_score(y_test, y_pred_gini)
    ]
})
final_metrics_gini


In [ ]:
# Task 18: same depth, but criterion='entropy' instead of default 'gini'
final_model_entropy = DecisionTreeClassifier(max_depth=best_depth, criterion='entropy', random_state=42)
final_model_entropy.fit(X_train, y_train)
y_pred_entropy = final_model_entropy.predict(X_test)

final_metrics_entropy = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-score'],
    'Baseline':            [acc, prec, rec, f1],
    'Chosen depth (gini)': [
        accuracy_score(y_test, y_pred_gini),
        precision_score(y_test, y_pred_gini),
        recall_score(y_test, y_pred_gini),
        f1_score(y_test, y_pred_gini)
    ],
    'Chosen depth (entropy)': [
        accuracy_score(y_test, y_pred_entropy),
        precision_score(y_test, y_pred_entropy),
        recall_score(y_test, y_pred_entropy),
        f1_score(y_test, y_pred_entropy)
    ]
})
final_metrics_entropy


**Tasks 17, 18  Compare the three models:**

 Compare the baseline, gini-at-chosen-depth, and
entropy-at-chosen-depth rows above. Usually gini and entropy give very
similar results (entropy is slightly more computationally expensive but
rarely changes performance much); note whether that held true for you, and
whether the chosen-depth model beat the baseline on the metrics you said
you trust most (Task 12).


In [ ]:
# Task 19: feature importances for your CHOSEN model
importances = pd.Series(final_model_gini.feature_importances_, index=X.columns)
top_features = importances.sort_values(ascending=False).head(10)

plt.figure(figsize=(7,5))
top_features.sort_values().plot(kind='barh')
plt.xlabel('Importance')
plt.title('Top 10 Feature Importances — Chosen Decision Tree')
plt.tight_layout()
plt.show()

top_features


**Task 19  Which 3–5 features matter most?**

 List the top 3–5 feature names from YOUR bar chart
above (e.g. `tenure`, `Contract_Two year`, `MonthlyCharges`, etc.  actual
names depend on your columns).

**Task 20  Connect to Lab 2 EDA:**

 Compare this ranking to what you found exploring
the data in Lab 2 (e.g. if you noticed churn was concentrated in
month-to-month contracts or low-tenure customers, and `tenure`/`Contract`
show up as top features here, say so explicitly — that agreement is a good
sanity check that the model learned something real rather than something
spurious).


## 8. Conclusion & Next Steps

**Task 21  Conclusion

Summarize: which model you landed on (baseline vs.
chosen depth, gini vs. entropy), its key metric values, and 1–2 concrete
limitations (e.g. still misses some churners, dataset imbalance, single
tree's instability compared to ensembles).

**Task 22  What would you try next?**

 e.g. trying an
ensemble method like Random Forest, handling class imbalance with
class_weight='balanced' or SMOTE, or engineering new features from tenure/
billing history.


## Task 23  ID3 from Scratch (Play Badminton Dataset)




In [ ]:
import math
from collections import Counter


data = [
    {'Outlook': 'Sunny',    'Temperature': 'Hot',  'Humidity': 'High',   'Wind': 'Weak',   'Play': 'No'},
    {'Outlook': 'Sunny',    'Temperature': 'Hot',  'Humidity': 'High',   'Wind': 'Strong', 'Play': 'No'},
    {'Outlook': 'Overcast', 'Temperature': 'Hot',  'Humidity': 'High',   'Wind': 'Weak',   'Play': 'Yes'},
    {'Outlook': 'Rain',     'Temperature': 'Mild', 'Humidity': 'High',   'Wind': 'Weak',   'Play': 'Yes'},
    {'Outlook': 'Rain',     'Temperature': 'Cool', 'Humidity': 'Normal', 'Wind': 'Weak',   'Play': 'Yes'},
    {'Outlook': 'Rain',     'Temperature': 'Cool', 'Humidity': 'Normal', 'Wind': 'Strong', 'Play': 'No'},
    {'Outlook': 'Overcast', 'Temperature': 'Cool', 'Humidity': 'Normal', 'Wind': 'Strong', 'Play': 'Yes'},
    {'Outlook': 'Sunny',    'Temperature': 'Mild', 'Humidity': 'High',   'Wind': 'Weak',   'Play': 'No'},
    {'Outlook': 'Sunny',    'Temperature': 'Cool', 'Humidity': 'Normal', 'Wind': 'Weak',   'Play': 'Yes'},
    {'Outlook': 'Rain',     'Temperature': 'Mild', 'Humidity': 'Normal', 'Wind': 'Weak',   'Play': 'Yes'},
    {'Outlook': 'Sunny',    'Temperature': 'Mild', 'Humidity': 'Normal', 'Wind': 'Strong', 'Play': 'Yes'},
    {'Outlook': 'Overcast', 'Temperature': 'Mild', 'Humidity': 'High',   'Wind': 'Strong', 'Play': 'Yes'},
    {'Outlook': 'Overcast', 'Temperature': 'Hot',  'Humidity': 'Normal', 'Wind': 'Weak',   'Play': 'Yes'},
    {'Outlook': 'Rain',     'Temperature': 'Mild', 'Humidity': 'High',   'Wind': 'Strong', 'Play': 'No'},
]

TARGET = 'Play'

def entropy(rows):
    """Shannon entropy of the target column for a set of rows."""
    counts = Counter(r[TARGET] for r in rows)
    total = len(rows)
    ent = 0.0
    for label, count in counts.items():
        p = count / total
        ent -= p * math.log2(p)
    return ent

def information_gain(rows, attribute):
    """Information gain of splitting `rows` on `attribute`."""
    total_entropy = entropy(rows)
    total = len(rows)


    subsets = {}
    for r in rows:
        subsets.setdefault(r[attribute], []).append(r)

    weighted_entropy = 0.0
    for value, subset in subsets.items():
        weighted_entropy += (len(subset) / total) * entropy(subset)

    return total_entropy - weighted_entropy

def majority_label(rows):
    return Counter(r[TARGET] for r in rows).most_common(1)[0][0]

def id3(rows, attributes, depth=0):
    labels = set(r[TARGET] for r in rows)


    if len(labels) == 1:
        return labels.pop()

    if not attributes:
        return majority_label(rows)

    print(f"{'  '*depth}Node with {len(rows)} rows, entropy={entropy(rows):.4f}")
    gains = {}
    for attr in attributes:
        g = information_gain(rows, attr)
        gains[attr] = g
        print(f"{'  '*depth}  Gain({attr}) = {g:.4f}")


    best_attr = max(gains, key=gains.get)
    print(f"{'  '*depth}--> Splitting on '{best_attr}'\n")

    tree = {best_attr: {}}
    remaining_attrs = [a for a in attributes if a != best_attr]

    values = set(r[best_attr] for r in rows)
    for v in values:
        subset = [r for r in rows if r[best_attr] == v]
        if not subset:
            tree[best_attr][v] = majority_label(rows)
        else:
            tree[best_attr][v] = id3(subset, remaining_attrs, depth + 1)

    return tree

attributes = ['Outlook', 'Temperature', 'Humidity', 'Wind']
tree = id3(data, attributes)

print("\nFinal ID3 tree (nested dict):")
print(tree)


**Note for Task 23:** the printed output above shows the information gain
of every attribute at every split, exactly as required — no scikit-learn or
other decision-tree library is used, only hand-written `entropy()` and
`information_gain()` functions.
